In [4]:
import requests
import pandas as pd

def fetch_elering_prices(start, end, area="ee"):
    url = "https://dashboard.elering.ee/api/nps/price"
    params = {
        "start": start.strftime("%Y-%m-%dT%H:%M:%S.000Z"),
        "end": end.strftime("%Y-%m-%dT%H:%M:%S.000Z"),
    }

    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    payload = r.json()

    data = payload.get("data", {})
    area_data = data.get(area)

    # Handle missing or empty result
    if not area_data:
        print(f"No data for {area} between {params['start']} and {params['end']}")
        return pd.DataFrame(columns=["timestamp", "price", "datetime"])

    df = pd.DataFrame(area_data)

    print("Columns returned:", df.columns.tolist())

    if "timestamp" not in df.columns:
        print(df.head())
        raise ValueError("Expected 'timestamp' column was not returned")

    df["datetime"] = pd.to_datetime(df["timestamp"], unit="s", utc=True)
    return df

now = pd.Timestamp.now(tz="UTC")
start0 = pd.Timestamp("2010-01-01", tz="UTC")

all_dfs = []
starts = pd.date_range(start0, now, freq="YS")

for start in starts:
    end = min(start + pd.DateOffset(years=1) - pd.Timedelta(seconds=1), now)
    chunk = fetch_elering_prices(start, end)

    if not chunk.empty:
        all_dfs.append(chunk)

df_all = pd.concat(all_dfs, ignore_index=True)
df_all = df_all.drop_duplicates(subset=["timestamp"])
df_all = df_all.sort_values("datetime").reset_index(drop=True)

print(df_all.head())
print(df_all.tail())

No data for ee between 2010-01-01T00:00:00.000Z and 2010-12-31T23:59:59.000Z
No data for ee between 2011-01-01T00:00:00.000Z and 2011-12-31T23:59:59.000Z
No data for ee between 2012-01-01T00:00:00.000Z and 2012-12-31T23:59:59.000Z
Columns returned: ['timestamp', 'price']
Columns returned: ['timestamp', 'price']
Columns returned: ['timestamp', 'price']
Columns returned: ['timestamp', 'price']
Columns returned: ['timestamp', 'price']
Columns returned: ['timestamp', 'price']
Columns returned: ['timestamp', 'price']
Columns returned: ['timestamp', 'price']
Columns returned: ['timestamp', 'price']
Columns returned: ['timestamp', 'price']
Columns returned: ['timestamp', 'price']
Columns returned: ['timestamp', 'price']
Columns returned: ['timestamp', 'price']
Columns returned: ['timestamp', 'price']
    timestamp  price                  datetime
0  1370210400  31.02 2013-06-02 22:00:00+00:00
1  1370214000  27.61 2013-06-02 23:00:00+00:00
2  1370217600  26.89 2013-06-03 00:00:00+00:00
3  1370

In [3]:
df_all

,timestamp,price,datetime
0,1370210400,31.02,2013-06-02 22:00:00+00:00
1,1370214000,27.61,2013-06-02 23:00:00+00:00
2,1370217600,26.89,2013-06-03 00:00:00+00:00
3,1370221200,26.62,2013-06-03 01:00:00+00:00
4,1370224800,28.13,2013-06-03 02:00:00+00:00
...,...,...,...
121387,1773999900,28.01,2026-03-20 09:45:00+00:00
121388,1774000800,28.11,2026-03-20 10:00:00+00:00
121389,1774001700,27.27,2026-03-20 10:15:00+00:00
121390,1774002600,25.84,2026-03-20 10:30:00+00:00
